# MinIO Connection Test

This notebook validates MinIO connectivity through `image_gallery.storage.MinioStorage`. Credentials are loaded from the repository root `.env` file.

In [ ]:
import os
from pathlib import Path
from urllib.parse import urlparse

from dotenv import load_dotenv

from image_gallery.storage import MinioStorage


repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
load_dotenv(repo_root / ".env")


def require_env(name: str) -> str:
    """读取必需环境变量，缺失时立即失败。"""
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"Missing required environment variable: {name}")
    return value


raw_endpoint = require_env("IMAGE_GALLERY_MINIO_ENDPOINT")
access_key = require_env("IMAGE_GALLERY_MINIO_ACCESS_KEY")
secret_key = require_env("IMAGE_GALLERY_MINIO_SECRET_KEY")
bucket = require_env("IMAGE_GALLERY_MINIO_BUCKET")

parsed = urlparse(raw_endpoint)
endpoint = parsed.netloc or raw_endpoint
secure = parsed.scheme == "https"

endpoint, secure, bucket

In [ ]:
storage = MinioStorage(storage_name="minio_test").connect(
    endpoint=endpoint,
    access_key=access_key,
    secret_key=secret_key,
    bucket=bucket,
    secure=secure,
)

print("connected bucket:", storage.bucket)
print("missing probe exists:", storage.exists("image_gallery_connection_probe_missing"))

## Read / Write Validation

Write a temporary object and two batch objects, read them back, compare payloads, then delete all test objects.

In [ ]:
from datetime import datetime, timezone

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
single_path = f"image_gallery_notebook_tests/{stamp}/single.txt"
batch_paths = [
    f"image_gallery_notebook_tests/{stamp}/batch-a.txt",
    f"image_gallery_notebook_tests/{stamp}/batch-b.txt",
]
single_payload = f"single payload {stamp}".encode()
batch_payloads = [b"batch payload a", b"batch payload b"]
created_paths = [single_path, *batch_paths]

try:
    single_uri = storage.write_bytes(single_path, single_payload, overwrite=False)
    single_read = storage.read_bytes(single_path)
    print("single uri:", single_uri)
    print("single read matches:", single_read == single_payload)
    print("single exists after write:", storage.exists(single_path))

    batch_write_results = storage.write_bytes(batch_paths, batch_payloads, overwrite=False)
    batch_read_results = storage.read_bytes(batch_paths)
    print("batch write ok:", [result.ok for result in batch_write_results])
    print("batch read ok:", [result.ok for result in batch_read_results])
    print(
        "batch read matches:",
        [result.value == expected for result, expected in zip(batch_read_results, batch_payloads, strict=True)],
    )
finally:
    for path in created_paths:
        if storage.exists(path):
            storage.delete(path)
    print("cleanup exists:", {path: storage.exists(path) for path in created_paths})